# Libraries

In [ ]:
import os
import random
import numpy as np
from PIL import Image
from tqdm import tqdm
import pydicom
import pandas as pd
from pydicom.multival import MultiValue
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import math
from torchvision import transforms, models
import csv
import timm
import torch.nn as nn
import torch.nn.functional as F
# optional: for AUC
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, matthews_corrcoef, roc_curve,
    brier_score_loss, balanced_accuracy_score, accuracy_score
)

In [ ]:
# TRAIN_CSV = "neh_train.csv"
# VAL_CSV   = "neh_val.csv"
# TEST_CSV  = "neh_test.csv"


TRAIN_CSV = "neh_train_reduced.csv"
VAL_CSV   = "neh_val_reduced.csv"
TEST_CSV  = "neh_test_reduced.csv"

# ===============================
# Task configuration
# ===============================
# NUM_CLASSES = 4
# CLASS_NAMES = ["NORMAL", "DRUSEN", "DME", "CNV"]

NUM_CLASSES = 2
CLASS_NAMES = ["NORMAL", "DISEASE"]


# Label meaning:
# four_class_label:
# 0 -> NORMAL
# 1 -> DRUSEN
# 2 -> DME
# 3 -> CNV

# binary_label:
# 0 -> NORMAL
# 1 -> DISEASE (DRUSEN / DME / CNV)

# ===============================
# Training configuration (initial)
# ===============================
IMG_SIZE = 512        # standard for ResNet18
BATCH_SIZE = 32
NUM_WORKERS = 4
NUM_EPOCHS = 30

LR = 5e-4
WEIGHT_DECAY = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
SEED = 42
set_seed(SEED)

# Dataset

In [ ]:
# ===============================
# Load CSVs
# ===============================
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("Train samples:", len(train_df))
print("Val samples  :", len(val_df))
print("Test samples :", len(test_df))

# ===============================
# Check required columns
# ===============================
required_cols = [
    "new_file_path",
    "label_text",
    "three_class_label",
    "binary_label",
    "patient_id"
]

for col in required_cols:
    assert col in train_df.columns, f"Missing column in train_df: {col}"
    assert col in val_df.columns,  f"Missing column in val_df: {col}"
    assert col in test_df.columns,  f"Missing column in test_df: {col}"

print("✅ All required columns present")

# ===============================
# Class distribution
# ===============================
print("\nTrain distribution:")
print(train_df["label_text"].value_counts())

print("\nValidation distribution:")
print(val_df["label_text"].value_counts())

print("\nTest distribution:")
print(test_df["label_text"].value_counts())

# ===============================
# Peek at data
# ===============================
train_df.head()


In [ ]:
# file_path = '/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/dhu_filtered.csv'
# import pandas as pd
# df = pd.read_csv(file_path)
# # 1. Identify unique patients and their specific category (AMD, DME, or NORMAL)
# # We use the first part of the patient_id string to identify the type
# patient_data = df.groupby('patient_id').first().reset_index()
# patient_data['category'] = patient_data['patient_id'].str.extract(r'([A-Za-z]+)')

# # Define the pool of patients
# all_patients = patient_data.copy()

# # 2. SELECT VALIDATION PATIENTS (1 Normal, 1 from AMD/DME pool)
# val_normal = all_patients[all_patients['category'] == 'NORMAL'].sample(1, random_state=SEED)
# val_abnormal = all_patients[all_patients['category'].isin(['AMD'])].sample(1, random_state=SEED)
# val_dme    = all_patients[all_patients['category'] == 'DME'].sample(1, random_state=42)

# val_list = pd.concat([val_normal, val_abnormal, val_dme])

# # Remove validation from pool
# pool_after_val = all_patients[~all_patients['patient_id'].isin(val_list['patient_id'])]

# # 3. SELECT TRAINING PATIENTS (1 Normal, 1 AMD, 1 DME)
# # Note: Using seed 42 as requested
# train_normal = pool_after_val[pool_after_val['category'] == 'NORMAL'].sample(1, random_state=42)
# train_amd    = pool_after_val[pool_after_val['category'] == 'AMD'].sample(1, random_state=42)
# train_dme    = pool_after_val[pool_after_val['category'] == 'DME'].sample(1, random_state=42)
# train_list   = pd.concat([train_normal, train_amd, train_dme])

# # 4. REMAINING ARE TEST
# test_list = pool_after_val[~pool_after_val['patient_id'].isin(train_list['patient_id'])]

# # 5. MAP BACK TO DATASET
# train_df = df[df['patient_id'].isin(train_list['patient_id'])].copy()
# val_df   = df[df['patient_id'].isin(val_list['patient_id'])].copy()
# test_df  = df[df['patient_id'].isin(test_list['patient_id'])].copy()

# # --- VERIFICATION ---
# print(f"Train Patients ({len(train_list)}): {train_list['patient_id'].tolist()}")
# print(f"Val Patients   ({len(val_list)}): {val_list['patient_id'].tolist()}")
# print(f"Test Patients  ({len(test_list)})")
# print("-" * 30)
# print(f"Image Counts -> Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

In [ ]:
# import pandas as pd
# from sklearn.model_selection import train_test_split

# # 1. Identify unique patients and their corresponding labels
# # We use the 'first' or 'mode' label for each patient to ensure stratification


# # 2. Load Dataset
# # file_path = '/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ucsd_filtered.csv'
# file_path = 'ucsd_filtered.csv'
# df = pd.read_csv(file_path)


# patient_data = df.groupby('patient_id')['binary_label'].agg(lambda x: x.mode()[0]).reset_index()

# # 2. Split Patients: 10% (Train+Val) and 90% (Test)
# train_val_patients, test_patients = train_test_split(
#     patient_data, 
#     test_size=0.80, 
#     stratify=patient_data['binary_label'], 
#     random_state=SEED
# )

# # 3. Split the 10% Patient chunk into 8% Train and 2% Val (0.02/0.10 = 0.20)
# train_patients, val_patients = train_test_split(
#     train_val_patients, 
#     test_size=0.20, 
#     stratify=train_val_patients['binary_label'], 
#     random_state=SEED
# )

# # 4. Map the split patients back to the original full dataframe
# train_df = df[df['patient_id'].isin(train_patients['patient_id'])].copy()
# val_df   = df[df['patient_id'].isin(val_patients['patient_id'])].copy()
# test_df  = df[df['patient_id'].isin(test_patients['patient_id'])].copy()

# # --- Verification ---
# print(f"Total Patients: {len(patient_data)}")
# print(f"Train Patients: {len(train_patients)} | Val Patients: {len(val_patients)} | Test Patients: {len(test_patients)}")
# print("-" * 30)
# print(f"Total Samples: {len(df)}")
# print(f"Train Samples: {len(train_df)} ({len(train_df)/len(df):.1%})")
# print(f"Val Samples:   {len(val_df)} ({len(val_df)/len(df):.1%})")
# print(f"Test Samples:  {len(test_df)} ({len(test_df)/len(df):.1%})")

# # Check for leakage (should be 0)
# overlap = set(train_df['patient_id']).intersection(set(test_df['patient_id']))
# print(f"\nPatient ID Overlap between Train and Test: {len(overlap)}")

In [ ]:
# import pandas as pd
# from sklearn.model_selection import train_test_split
# import numpy as np
# import random
# import torch

# # 1. Set Seed for Reproducibility
# def set_seed(seed=42):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     if torch.cuda.is_available():
#         torch.cuda.manual_seed_all(seed)
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

# SEED = 42
# set_seed(SEED)

# # 2. Load Dataset
# file_path = '/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/octC8_filtered.csv'
# df = pd.read_csv(file_path)

# # 3. Split the entire data into 10% (for train+val) and 90% (for test)
# # This ensures 90% of the total data goes to test_df
# train_val_df, test_df = train_test_split(
#     df, 
#     test_size=0.80, 
#     stratify=df['binary_label'], 
#     random_state=SEED
# )

# # 4. Split the 10% chunk into 8% Train and 2% Val
# # Note: 2% of total is 20% of this 10% chunk (0.02 / 0.10 = 0.20)
# train_df, val_df = train_test_split(
#     train_val_df, 
#     test_size=0.20, 
#     stratify=train_val_df['binary_label'], 
#     random_state=SEED
# )

# # Verification
# print(f"Total samples: {len(df)}")
# print(f"Train size (8%): {len(train_df)}")
# print(f"Val size   (2%): {len(val_df)}")
# print(f"Test size (90%): {len(test_df)}")

# # Check label distribution
# print("\nLabel Distribution (should be ~52.7% label 1):")
# print(f"Train: \n{train_df['binary_label'].value_counts()}")
# print(f"Val: \n{val_df['binary_label'].value_counts()}")

In [ ]:
print(train_df['binary_label'].value_counts())
print(val_df['binary_label'].value_counts())
print(test_df['binary_label'].value_counts())

In [ ]:
from PIL import Image, ImageFile
import torch

ImageFile.LOAD_TRUNCATED_IMAGES = True

class NEHOCTDataset(torch.utils.data.Dataset):
    def __init__(self, df, transform=None, label_col="binary_label"):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        try:
            img = Image.open(row["new_file_path"])

            # Ensure 3-channel RGB
            if img.mode != "RGB":
                img = img.convert("RGB")

        except Exception as e:
            raise RuntimeError(
                f"Failed to load image: {row['new_file_path']}"
            ) from e

        label = int(row[self.label_col])

        # if self.transform:
        #     img = self.transform(img)

        return img, label


def pil_collate_fn(batch):
    """
    batch: list of (PIL_image, label)
    returns:
        images: list[PIL.Image]
        labels: torch.LongTensor
    """
    images, labels = zip(*batch)
    labels = torch.tensor(labels, dtype=torch.long)
    return list(images), labels

# class AddGaussianNoise(object):
#     def __init__(self, mean=0.0, std=0.02):
#         self.mean = mean
#         self.std = std

#     def __call__(self, tensor):
#         noise = torch.randn_like(tensor) * self.std + self.mean
#         return torch.clamp(tensor + noise, 0.0, 1.0)

# class AddSpeckleNoise(object):
#     def __init__(self, std=0.02):
#         self.std = std

#     def __call__(self, tensor):
#         noise = torch.randn_like(tensor) * self.std
#         return torch.clamp(tensor + tensor * noise, 0.0, 1.0)

class AddSpeckleNoise(object):
    def __init__(self, std=0.01):
        self.std = std

    def __call__(self, tensor):
        if self.std <= 0:
            return tensor
        noise = torch.randn_like(tensor) * self.std
        out = tensor + tensor * noise
        return torch.clamp(out, 0.0, 1.0)



In [ ]:
# train_transform = transforms.Compose([
#     transforms.Resize((IMG_SIZE, IMG_SIZE)),
#     transforms.RandomHorizontalFlip(),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                          std=[0.229, 0.224, 0.225]),
# ])

# train_transform_sub = transforms.Compose([
#     transforms.Resize((IMG_SIZE, IMG_SIZE)),
#     transforms.RandomResizedCrop(
#         IMG_SIZE,
#         scale=(0.9, 1.0),
#         ratio=(0.95, 1.05)
#     ),
#     transforms.RandomHorizontalFlip(p=0.5),

#     transforms.ToTensor(),

#     # AddGaussianNoise(std=0.02),
#     AddSpeckleNoise(std=0.02),

#     transforms.Normalize(
#         mean=[0.485, 0.456, 0.406],
#         std=[0.229, 0.224, 0.225]
#     ),
# ])

# val_transform = transforms.Compose([
#     transforms.Resize((IMG_SIZE, IMG_SIZE)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                          std=[0.229, 0.224, 0.225]),
# ])

# test_transform = transforms.Compose([
#     transforms.Resize((IMG_SIZE, IMG_SIZE)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                          std=[0.229, 0.224, 0.225]),
# ])
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    # Safe for OCT + Mammogram
    transforms.RandomHorizontalFlip(p=0.5),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

train_transform_sub = transforms.Compose([
    transforms.RandomChoice([
        transforms.Resize((int(0.9 * IMG_SIZE), int(0.9 * IMG_SIZE))),
        transforms.Resize((int(0.85 * IMG_SIZE), int(0.85 * IMG_SIZE))),
        transforms.Resize((int(0.8 * IMG_SIZE), int(0.8 * IMG_SIZE))),
    ]),

    # Restore to network input size
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.ToTensor(),

    transforms.RandomApply(
        [AddSpeckleNoise(std=0.01)],
        p=0.3
    ),
    
    # AddSpeckleNoise(std=0.01),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])


In [ ]:
train_dataset = NEHOCTDataset(train_df, label_col='binary_label')
val_dataset   = NEHOCTDataset(val_df, label_col='binary_label')
test_dataset  = NEHOCTDataset(test_df, label_col='binary_label')
ucsd_dataset  = NEHOCTDataset(df, label_col='binary_label')

pinmem = True if torch.cuda.is_available() else False

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=pinmem, collate_fn=pil_collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pinmem, collate_fn=pil_collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pinmem, collate_fn=pil_collate_fn)
test_loader = DataLoader(ucsd_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=pinmem, collate_fn=pil_collate_fn)

In [ ]:
# next(iter(train_loader))


In [ ]:
# import matplotlib.pyplot as plt

# IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
# IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# def denormalize(img):
#     """
#     img: Tensor [3, H, W] after Normalize
#     """
#     return img * IMAGENET_STD + IMAGENET_MEAN
# def visualize_dataloader_batch(
#     dataloader,
#     transform,
#     num_images=4,
#     title="DataLoader Visualization",
# ):
#     """
#     Visualize images exactly as seen by the model AFTER transform.

#     dataloader : DataLoader with pil_collate_fn
#     transform  : torchvision transform to apply (train / sub / val / test)
#     """
#     imgs_pil, labels = next(iter(dataloader))

#     num_images = min(num_images, len(imgs_pil))

#     plt.figure(figsize=(4 * num_images, 4))

#     for i in range(num_images):
#         # Apply transform (this is what the model sees)
#         img = transform(imgs_pil[i])  # Tensor [3,H,W]

#         # Denormalize
#         img = img.cpu()
#         img = img * IMAGENET_STD + IMAGENET_MEAN
#         img = img.clamp(0, 1)

#         plt.subplot(1, num_images, i + 1)
#         plt.imshow(img.permute(1, 2, 0))
#         plt.title(f"Label: {labels[i].item()}")
#         plt.axis("off")

#     plt.suptitle(title, fontsize=14)
#     plt.tight_layout()
#     plt.show()

In [ ]:
# visualize_dataloader_batch(
#     train_loader,
#     transform=train_transform_sub,
#     num_images=8,
#     title="Train – GradAug Sub-network View"
# )

# Metrics

In [ ]:
def compute_uncertainty_stats(y_proba: np.ndarray, y_true: np.ndarray):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    y_true  = np.asarray(y_true, dtype=np.int64)

    y_proba = np.nan_to_num(y_proba, nan=0.5, posinf=1.0, neginf=0.0)

    eps = 1e-8
    p1 = np.clip(y_proba, eps, 1.0 - eps)
    p0 = 1.0 - p1

    entropy = -(p0 * np.log(p0) + p1 * np.log(p1))
    confidence = np.maximum(p0, p1)
    uncertainty = 1.0 - confidence

    out = {
        "avg_entropy": float(np.mean(entropy)),
        "entropy_std": float(np.std(entropy)),
        "avg_uncertainty": float(np.mean(uncertainty)),
        "uncertainty_std": float(np.std(uncertainty)),
    }

    # ---- class-conditional statistics ----
    for cls in (0, 1):
        mask = (y_true == cls)

        if mask.any():
            out[f"entropy_class{cls}_avg"] = float(entropy[mask].mean())
            out[f"entropy_class{cls}_std"] = float(entropy[mask].std())
            out[f"uncertainty_class{cls}_avg"] = float(uncertainty[mask].mean())
            out[f"uncertainty_class{cls}_std"] = float(uncertainty[mask].std())
        else:
            out[f"entropy_class{cls}_avg"] = float("nan")
            out[f"entropy_class{cls}_std"] = float("nan")
            out[f"uncertainty_class{cls}_avg"] = float("nan")
            out[f"uncertainty_class{cls}_std"] = float("nan")

    return out

In [ ]:
def compute_binary_class_weights(df):
    counts = df["binary_label"].value_counts().sort_index().values
    total = counts.sum()
    weights = total / (2.0 * counts)
    return torch.tensor(weights, dtype=torch.float)

class_weights = compute_binary_class_weights(train_df).to(DEVICE)

print("Class counts:")
print(train_df["binary_label"].value_counts().sort_index())

print("Class weights:", class_weights.cpu().numpy())

In [ ]:
# import csv
# import os

# CSV_HEADER = [
#     "epoch",
#     "train_loss", "val_loss", "test_loss",
#     "train_acc", "val_acc", "test_acc",
#     "train_auc", "val_auc", "test_auc",
#     "val_pr_auc", "test_pr_auc",
#     "val_f1", "test_f1", "val_macro_f1", "test_macro_f1",
#     "val_precision", "val_recall", "val_npv",
#     "test_precision", "test_recall", "test_npv",
#     "val_specificity", "test_specificity",
#     "val_sens_at_spec_90", "test_sens_at_spec_90",
#     "avg_entropy", "entropy_std",
#     "avg_uncertainty", "uncertainty_std",
#     "entropy_class0_avg", "entropy_class0_std",
#     "entropy_class1_avg", "entropy_class1_std",
#     "uncertainty_class0_avg", "uncertainty_class0_std",
#     "uncertainty_class1_avg", "uncertainty_class1_std",
#     "tn", "fp", "fn", "tp", "n_samples"
# ]

# def append_metrics_to_csv(csv_path, row_dict, float_fmt="{:.6f}"):
#     """
#     Appends one epoch of metrics to CSV.
#     Floats are formatted to fixed precision (default .6f).
#     Creates file + header if missing.
#     """
#     file_exists = os.path.isfile(csv_path)

#     formatted_row = {}
#     for k, v in row_dict.items():
#         if isinstance(v, float):
#             if np.isnan(v):
#                 formatted_row[k] = np.nan
#             else:
#                 formatted_row[k] = float_fmt.format(v)
#         else:
#             formatted_row[k] = v

#     with open(csv_path, mode="a", newline="") as f:
#         writer = csv.DictWriter(f, fieldnames=CSV_HEADER)

#         if not file_exists:
#             writer.writeheader()

#         writer.writerow({k: formatted_row.get(k, np.nan) for k in CSV_HEADER})

In [ ]:
import csv
import os

CSV_HEADER = [
    "epoch",
    "train_loss", "val_loss",
    "train_acc", "val_acc", 
    "train_auc", "val_auc",
    "val_f1", "val_macro_f1",  
    # "avg_entropy", "entropy_std",
    # "avg_uncertainty", "uncertainty_std",
    # "entropy_class0_avg", "entropy_class0_std",
    # "entropy_class1_avg", "entropy_class1_std",
    # "uncertainty_class0_avg", "uncertainty_class0_std",
    # "uncertainty_class1_avg", "uncertainty_class1_std",
    # "tn", "fp", "fn", "tp", "n_samples"
]

def append_metrics_to_csv(csv_path, row_dict, float_fmt="{:.6f}"):
    """
    Appends one epoch of metrics to CSV.
    Floats are formatted to fixed precision (default .6f).
    Creates file + header if missing.
    """
    file_exists = os.path.isfile(csv_path)

    formatted_row = {}
    for k, v in row_dict.items():
        if isinstance(v, float):
            if np.isnan(v):
                formatted_row[k] = np.nan
            else:
                formatted_row[k] = float_fmt.format(v)
        else:
            formatted_row[k] = v

    with open(csv_path, mode="a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_HEADER)

        if not file_exists:
            writer.writeheader()

        writer.writerow({k: formatted_row.get(k, np.nan) for k in CSV_HEADER})

In [ ]:
def compute_binary_metrics(y_proba, y_true, threshold=0.5, target_spec=0.90):
    """
    y_proba: list or np.array of predicted probability for class 1
    y_true:  list or np.array of integer {0,1}
    threshold: float for converting proba -> label
    returns: dict of metrics including confusion matrix entries
    """
    y_proba = np.asarray(y_proba)
    y_true = np.asarray(y_true).astype(int)

    out = {}
    # ROC-AUC and PR-AUC (handle single-class edge)
    try:
        out['roc_auc'] = float(roc_auc_score(y_true, y_proba)) if len(np.unique(y_true)) > 1 else float("nan")
    except Exception:
        out['roc_auc'] = float("nan")
    try:
        out['pr_auc'] = float(average_precision_score(y_true, y_proba)) if len(np.unique(y_true)) > 1 else float("nan")
    except Exception:
        out['pr_auc'] = float("nan")

    # Binary predictions
    y_pred = (y_proba >= threshold).astype(int)

    # Standard metrics
    out['precision'] = float(precision_score(y_true, y_pred, average="macro", zero_division=0))
    out['recall'] = float(recall_score(y_true, y_pred, average="macro", zero_division=0))    # sensitivity
    out['f1'] = float(f1_score(y_true, y_pred, zero_division=0))
    out['macro_f1'] = float(
        f1_score(y_true, y_pred, average="macro", zero_division=0)
    )
    out['balanced_acc'] = float(balanced_accuracy_score(y_true, y_pred))
    # MCC (may raise if degenerate)
    # try:
    #     out['mcc'] = float(matthews_corrcoef(y_true, y_pred))
    # except Exception:
    #     out['mcc'] = float("nan")

    # Confusion matrix
    try:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
        out['tn'] = int(tn); out['fp'] = int(fp); out['fn'] = int(fn); out['tp'] = int(tp)
        out['specificity'] = float(tn / (tn + fp)) if (tn + fp) > 0 else float("nan")
        out['sensitivity'] = float(tp / (tp + fn)) if (tp + fn) > 0 else float("nan")
        out['npv'] = float(tn / (tn + fn)) if (tn + fn) > 0 else float("nan")

    except Exception:
        out['tn'] = out['fp'] = out['fn'] = out['tp'] = 0
        out['specificity'] = out['sensitivity'] = float("nan")

    # Brier score
    try:
        out['brier'] = float(brier_score_loss(y_true, y_proba))
    except Exception:
        out['brier'] = float("nan")

    out['n_samples'] = int(len(y_true))
    out['threshold'] = float(threshold)
    try:
        thresh_list = np.linspace(0, 1, 200)
        best_sens = float("nan")
        best_thr = float("nan")

        for thr in thresh_list:
            yp = (y_proba >= thr).astype(int)
            cm = confusion_matrix(y_true, yp, labels=[0,1])
            tn, fp, fn, tp = cm.ravel()
            spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
            sens = tp / (tp + fn) if (tp + fn) > 0 else float("nan")

            if not np.isnan(spec) and spec >= target_spec:
                # choose largest sensitivity among valid thresholds
                if np.isnan(best_sens) or sens > best_sens:
                    best_sens = sens
                    best_thr = thr

        out["sens_at_spec"] = best_sens
        out["thr_at_spec"] = best_thr
        out["target_spec"] = target_spec

    except Exception:
        out["sens_at_spec"] = float("nan")
        out["thr_at_spec"] = float("nan")
        out["target_spec"] = target_spec
    
    try:
        u_stats = compute_uncertainty_stats(y_proba, y_true)
        out.update(u_stats)
    except Exception:
        for k in [
            "avg_entropy","entropy_std",
            "avg_uncertainty","uncertainty_std",
            "entropy_class0_avg","entropy_class0_std",
            "entropy_class1_avg","entropy_class1_std",
            "uncertainty_class0_avg","uncertainty_class0_std",
            "uncertainty_class1_avg","uncertainty_class1_std",
        ]:
            out[k] = float("nan")
    return out

def print_epoch_summary(epoch, train_loss, val_loss, test_loss, train_auc, val_auc, test_auc, train_acc, val_acc, val_macro_f1):
    print("=" * 90)
    print(f"Epoch {epoch}")
    print("-" * 90)
    print(f"Train | Loss={(train_loss):.4f}  AUC={(train_auc):.4f}  Acc={(train_acc):.4f}")
    print(f"Val   | Loss={(val_loss):.4f}  AUC={(val_auc):.4f}  Acc={(val_acc):.4f}")
    print("Val Macro F1: {:.4f}".format(val_macro_f1))

    print("\nClassification Metrics:")
    print("=" * 90)

In [ ]:
# def compute_binary_metrics(y_proba, y_true, threshold=0.5, target_spec=0.90):
#     """
#     y_proba: list or np.array of predicted probability for class 1
#     y_true:  list or np.array of integer {0,1}
#     threshold: float for converting proba -> label
#     returns: dict of metrics including confusion matrix entries
#     """
#     y_proba = np.asarray(y_proba)
#     y_true = np.asarray(y_true).astype(int)

#     out = {}
#     # ROC-AUC and PR-AUC (handle single-class edge)
#     try:
#         out['roc_auc'] = float(roc_auc_score(y_true, y_proba)) if len(np.unique(y_true)) > 1 else float("nan")
#     except Exception:
#         out['roc_auc'] = float("nan")
#     try:
#         out['pr_auc'] = float(average_precision_score(y_true, y_proba)) if len(np.unique(y_true)) > 1 else float("nan")
#     except Exception:
#         out['pr_auc'] = float("nan")

#     # Binary predictions
#     y_pred = (y_proba >= threshold).astype(int)

#     # Standard metrics
#     out['precision'] = float(precision_score(y_true, y_pred, average="macro", zero_division=0))
#     out['recall'] = float(recall_score(y_true, y_pred, average="macro", zero_division=0))    # sensitivity
#     out['f1'] = float(f1_score(y_true, y_pred, zero_division=0))
#     out['macro_f1'] = float(
#         f1_score(y_true, y_pred, average="macro", zero_division=0)
#     )
#     out['balanced_acc'] = float(balanced_accuracy_score(y_true, y_pred))
#     # MCC (may raise if degenerate)
#     # try:
#     #     out['mcc'] = float(matthews_corrcoef(y_true, y_pred))
#     # except Exception:
#     #     out['mcc'] = float("nan")

#     # Confusion matrix
#     try:
#         tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
#         out['tn'] = int(tn); out['fp'] = int(fp); out['fn'] = int(fn); out['tp'] = int(tp)
#         out['specificity'] = float(tn / (tn + fp)) if (tn + fp) > 0 else float("nan")
#         out['sensitivity'] = float(tp / (tp + fn)) if (tp + fn) > 0 else float("nan")
#         out['npv'] = float(tn / (tn + fn)) if (tn + fn) > 0 else float("nan")

#     except Exception:
#         out['tn'] = out['fp'] = out['fn'] = out['tp'] = 0
#         out['specificity'] = out['sensitivity'] = float("nan")

#     # Brier score
#     try:
#         out['brier'] = float(brier_score_loss(y_true, y_proba))
#     except Exception:
#         out['brier'] = float("nan")

#     out['n_samples'] = int(len(y_true))
#     out['threshold'] = float(threshold)
#     try:
#         thresh_list = np.linspace(0, 1, 200)
#         best_sens = float("nan")
#         best_thr = float("nan")

#         for thr in thresh_list:
#             yp = (y_proba >= thr).astype(int)
#             cm = confusion_matrix(y_true, yp, labels=[0,1])
#             tn, fp, fn, tp = cm.ravel()
#             spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
#             sens = tp / (tp + fn) if (tp + fn) > 0 else float("nan")

#             if not np.isnan(spec) and spec >= target_spec:
#                 # choose largest sensitivity among valid thresholds
#                 if np.isnan(best_sens) or sens > best_sens:
#                     best_sens = sens
#                     best_thr = thr

#         out["sens_at_spec"] = best_sens
#         out["thr_at_spec"] = best_thr
#         out["target_spec"] = target_spec

#     except Exception:
#         out["sens_at_spec"] = float("nan")
#         out["thr_at_spec"] = float("nan")
#         out["target_spec"] = target_spec
    
#     try:
#         u_stats = compute_uncertainty_stats(y_proba, y_true)
#         out.update(u_stats)
#     except Exception:
#         for k in [
#             "avg_entropy","entropy_std",
#             "avg_uncertainty","uncertainty_std",
#             "entropy_class0_avg","entropy_class0_std",
#             "entropy_class1_avg","entropy_class1_std",
#             "uncertainty_class0_avg","uncertainty_class0_std",
#             "uncertainty_class1_avg","uncertainty_class1_std",
#         ]:
#             out[k] = float("nan")
#     return out

# def print_epoch_summary(epoch, train_loss, val_loss, test_loss, train_auc, val_auc, test_auc, train_acc, val_acc, test_acc, metrics, uncert):
#     print("=" * 90)
#     print(f"Epoch {epoch}")
#     print("-" * 90)
#     print(f"Train | Loss={(train_loss):.4f}  AUC={(train_auc):.4f}  Acc={(train_acc):.4f}")
#     print(f"Val   | Loss={(val_loss):.4f}  AUC={(val_auc):.4f}  Acc={(val_acc):.4f}")
#     print(f"Test  | Loss={(test_loss):.4f}  AUC={(test_auc):.4f}  Acc={(test_acc):.4f}")

#     print("\nClassification Metrics:")
#     print(
#         f"  ROC-AUC={(metrics['roc_auc']):.4f}  PR-AUC={(metrics['pr_auc']):.4f}  Brier_score={(metrics['brier']):.4f}  "
#         f"F1={(metrics['f1']):.4f}  Macro-F1={(metrics['macro_f1']):.4f}  Precision={(metrics['precision']):.4f}  Recall={(metrics['recall']):.4f}  NPV={(metrics['npv']):.4f}" f"  Specificity={(metrics['specificity']):.4f}  "  f"Sens@Spec90={(metrics['sens_at_spec']):.4f} ")

#     print("\nUncertainty Metrics:")
#     print(
#         f"  AvgEntropy={(uncert['avg_entropy']):.4f} ± {(uncert['entropy_std']):.4f} | " f"AvgUncertainty={(uncert['avg_uncertainty']):.4f} ± {(uncert['uncertainty_std']):.4f}" f"  Entropy(C0)={(uncert['entropy_class0_avg']):.4f}  " f"Entropy(C1)={(uncert['entropy_class1_avg']):.4f}" f"  Uncertainty(C0)={(uncert['uncertainty_class0_avg']):.4f}  " f"Uncertainty(C1)={(uncert['uncertainty_class1_avg']):.4f}"
#     )

#     print("\nConfusion Matrix:" f"  TP={metrics['tp']}  FP={metrics['fp']} " f"FN={metrics['fn']}  TN={metrics['tn']}")
#     print("=" * 90)

# Model

In [ ]:
class ResNet34GradAug(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()
        base = models.resnet34(pretrained=pretrained)

        self.conv1 = base.conv1
        self.bn1 = base.bn1
        self.relu = base.relu
        self.maxpool = base.maxpool

        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4   # 🔑 always used

        self.avgpool = base.avgpool
        self.fc = nn.Linear(512, num_classes)

        # number of blocks before layer4
        self.n_pre_blocks = (
            len(self.layer1)
            + len(self.layer2)
            + len(self.layer3)
        )

    def forward_full(self, x):
        x = self._forward_stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self._head(x)

    def forward(self, x):
        return self.forward_full(x)

    def forward_subnet(self, x, keep_ratio):
        """
        keep_ratio applies ONLY to blocks before layer4
        """
        n_keep = int(self.n_pre_blocks * keep_ratio)
        n_keep = max(1, n_keep)

        x = self._forward_stem(x)

        blocks = []
        for layer in [self.layer1, self.layer2, self.layer3]:
            for block in layer:
                blocks.append(block)

        for i in range(n_keep):
            x = blocks[i](x)

        x = self.layer4(x)   # 🔑 ALWAYS include
        return self._head(x)

    def _forward_stem(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        return x

    def _head(self, x):
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

def sample_keep_ratio():
    return np.random.uniform(0.8, 0.99)

model = ResNet34GradAug(num_classes=NUM_CLASSES, pretrained=True).to(DEVICE)
model = model.to(DEVICE)

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights).to(DEVICE)

optimizer = torch.optim.AdamW(
    [
        # Stem
        {"params": model.conv1.parameters(), "lr": 1e-4},
        {"params": model.bn1.parameters(),   "lr": 1e-4},

        # Residual layers
        {"params": model.layer1.parameters(), "lr": 1e-4},
        {"params": model.layer2.parameters(), "lr": 1e-4},
        {"params": model.layer3.parameters(), "lr": 1e-4},
        {"params": model.layer4.parameters(), "lr": 1e-4},

        # Classification head
        {"params": model.fc.parameters(),     "lr": 3e-4},
    ],
    weight_decay=1e-4
)
# optimizer = torch.optim.AdamW(
#     [
#         # Stem
#         {"params": model.conv1.parameters(), "lr": 5e-4},
#         {"params": model.bn1.parameters(),   "lr": 5e-4},

#         # Residual layers
#         {"params": model.layer1.parameters(), "lr": 5e-4},
#         {"params": model.layer2.parameters(), "lr": 5e-4},
#         {"params": model.layer3.parameters(), "lr": 5e-4},
#         {"params": model.layer4.parameters(), "lr": 5e-4},

#         # Classification head
#         {"params": model.fc.parameters(),     "lr": 7e-4},
#     ],
#     weight_decay=1e-4
# )

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-6)

def kl_loss(student_logits, teacher_logits):
    return F.kl_div(
        F.log_softmax(student_logits, dim=1),
        F.softmax(teacher_logits.detach(), dim=1),
        reduction="batchmean"
    )

def set_bn_running_stats(model, flag):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.track_running_stats = flag

# Loops

In [ ]:
def train(
    model,
    loader,
    optimizer,
    device,
    criterion,
    train_tf_full,
    train_tf_sub,
    epoch,
    n_views=2,
    lambda_kl=0.02,
    warmup_epochs=5
):
    model.train()
    epoch_loss = 0.0
    probs, targets = [], []

    kl_weight = lambda_kl * min(1.0, epoch / 10)
    
    for imgs_pil, y in tqdm(loader, desc="Train", leave=False):
        y = y.to(device)
        set_bn_running_stats(model, True)
        # -------- Teacher (full net) --------
        imgs_full = torch.stack([train_tf_full(img) for img in imgs_pil]).to(device)

        optimizer.zero_grad()
        logits_full = model.forward_full(imgs_full)
        loss = criterion(logits_full, y)

        teacher_probs = F.softmax(logits_full.detach(), dim=1)
        teacher_probs = torch.clamp(teacher_probs, 1e-4, 1.0 - 1e-4)

        # -------- Subnets --------
        if epoch > warmup_epochs:
            set_bn_running_stats(model, False)

            for _ in range(n_views):
                keep_ratio = sample_keep_ratio()

                imgs_sub = torch.stack(
                    [train_tf_sub(img) for img in imgs_pil]
                ).to(device)

                with torch.enable_grad():
                    logits_sub = model.forward_subnet(imgs_sub, keep_ratio)

                loss_kl = F.kl_div(
                    F.log_softmax(logits_sub, dim=1),
                    teacher_probs,
                    reduction="batchmean"
                )

                loss += (kl_weight / n_views) * loss_kl
            set_bn_running_stats(model, True)
       
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        epoch_loss += loss.item()
        probs.extend(teacher_probs[:, 1].cpu().numpy())
        targets.extend(y.cpu().numpy())

    epoch_loss /= len(loader)
    auc = roc_auc_score(targets, probs)
    acc = np.mean((np.array(probs) >= 0.5) == np.array(targets))
    return epoch_loss, auc, acc

In [ ]:
@torch.no_grad()
def validate(
    model,
    loader,
    device,
    criterion,
    val_transform
):
    model.eval()

    probs, targets = [], []
    epoch_loss = 0.0
    c=0
    for imgs_pil, y in tqdm(loader, desc="Val", leave=False):
        y = y.to(device)

        imgs = torch.stack(
            [val_transform(img) for img in imgs_pil]
        ).to(device)

        logits = model.forward_full(imgs)
        loss = criterion(logits, y)

        p = F.softmax(logits, dim=1)

        epoch_loss += loss.item()
        probs.extend(p[:, 1].cpu().numpy())
        targets.extend(y.cpu().numpy())
       
    epoch_loss /= len(loader)
    
    metrics = compute_binary_metrics(probs, targets)
    acc = np.mean((np.array(probs) >= 0.5) == np.array(targets))
    auc = roc_auc_score(targets, probs)

    return epoch_loss, metrics, acc, auc, probs, targets


In [ ]:
@torch.no_grad()
def test(
    model,
    loader,
    device,
    criterion,
    test_transform
):
    model.eval()

    probs, targets = [], []
    epoch_loss = 0.0
    
    for imgs_pil, y in tqdm(loader, desc="Test", leave=False):
        y = y.to(device)

        imgs = torch.stack(
            [test_transform(img) for img in imgs_pil]
        ).to(device)

        logits = model.forward_full(imgs)
        loss = criterion(logits, y)

        p = F.softmax(logits, dim=1)

        epoch_loss += loss.item()
        probs.extend(p[:, 1].cpu().numpy())
        targets.extend(y.cpu().numpy())

    epoch_loss /= len(loader)
    
    metrics = compute_binary_metrics(probs, targets)
    acc = np.mean((np.array(probs) >= 0.5) == np.array(targets))
    auc = roc_auc_score(targets, probs)
    uncert = compute_uncertainty_stats(probs, targets)
    
    return epoch_loss, metrics, acc, auc, probs, targets, uncert


In [ ]:
def set_bn_eval(m):
    if isinstance(m, nn.BatchNorm2d):
        m.eval()
        
def train_head_only(
    model,
    loader,
    optimizer,
    device,
    criterion,
    train_tf_full,
):
    model.train()

    model.apply(set_bn_eval)


    epoch_loss = 0.0
    probs, targets = [], []

    for imgs_pil, y in tqdm(loader, desc="Train", leave=False):
        y = y.to(device)

        imgs = torch.stack(
            [train_tf_full(img) for img in imgs_pil]
        ).to(device)

        optimizer.zero_grad()

        # 🔑 ALWAYS full forward
        logits = model.forward_full(imgs)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        prob = F.softmax(logits, dim=1)[:, 1]
        probs.extend(prob.detach().cpu().numpy())
        targets.extend(y.cpu().numpy())

    epoch_loss /= len(loader)
    auc = roc_auc_score(targets, probs)
    acc = np.mean((np.array(probs) >= 0.5) == np.array(targets))

    return epoch_loss, auc, acc


@torch.no_grad()
def validate_head_only(
    model,
    loader,
    device,
    criterion,
    val_tf,
):
    model.eval()
    model.apply(set_bn_eval)


    epoch_loss = 0.0
    probs, targets = [], []

    for imgs_pil, y in tqdm(loader, desc="Val", leave=False):
        y = y.to(device)

        imgs = torch.stack(
            [val_tf(img) for img in imgs_pil]
        ).to(device)

        logits = model.forward_full(imgs)
        loss = criterion(logits, y)

        epoch_loss += loss.item()

        prob = F.softmax(logits, dim=1)[:, 1]
        probs.extend(prob.cpu().numpy())
        targets.extend(y.cpu().numpy())

    epoch_loss /= len(loader)
    auc = roc_auc_score(targets, probs)
    acc = np.mean((np.array(probs) >= 0.5) == np.array(targets))
    macro_f1 = f1_score(targets, (np.array(probs) >= 0.5).astype(int), average="macro", zero_division=0)

    return epoch_loss, auc, acc, macro_f1



In [ ]:
def aggregate_uncertainty_stats(stats_list):
    if len(stats_list) == 0:
        return {}

    keys = stats_list[0].keys()
    out = {}

    for k in keys:
        vals = [d[k] for d in stats_list]
        out[k] = float(np.mean(vals))

    return out

# Training

In [ ]:
models_path = "ResNet34_neh_reduced_5e4_GradAug_withjust_with_only_speckle"
os.makedirs(models_path, exist_ok=True)
log_file = os.path.join(models_path, "training_log.csv")

BEST_MACRO_F1 = -1.0
WARMUP_EPOCHS = 3          # IMPORTANT
LAMBDA_KL = 0.03

# checkpoint_path = "/media/miglab/DATA_20TB1/Uncertainty/ResNet34+GradAug_4096_balanced/epoch_2.pth"
# checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

# model.load_state_dict(checkpoint["state_dict"])
# model.to(DEVICE)

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")

    # -------------------------
    # Train
    # -------------------------
    train_loss, train_auc, train_acc = train(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        device=DEVICE,
        criterion=criterion,
        train_tf_full=train_transform,
        train_tf_sub=train_transform_sub,
        epoch=epoch,
        n_views=2,
        lambda_kl=LAMBDA_KL,
        warmup_epochs=WARMUP_EPOCHS
    )

    # -------------------------
    # Validate
    # -------------------------
    val_loss, val_metrics, val_acc, val_auc, val_probs, val_targets = validate(
        model=model,
        loader=val_loader,
        device=DEVICE,
        criterion=criterion,
        val_transform=val_transform
    )

    test_loss, test_metrics, test_acc, test_auc, test_probs, test_targets, test_uncert = test(
        model=model,
        loader=test_loader,
        device=DEVICE,
        criterion=criterion,
        val_transform=test_transform
    )

    scheduler.step()

    # -------------------------
    # Save checkpoint
    # -------------------------
    ckpt = {
        "epoch": epoch,
        "state_dict": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "val_auc": val_metrics["roc_auc"]
    }

    torch.save(ckpt, os.path.join(models_path, f"epoch_{epoch}.pth"))

    if val_metrics["macro_f1"] > BEST_MACRO_F1:
        BEST_MACRO_F1 = val_metrics["macro_f1"]
        torch.save(ckpt, os.path.join(models_path, "best_model.pth"))

    # -------------------------
    # CSV logging
    # -------------------------
    row = {
        "epoch": epoch,
        "train_loss": train_loss, "val_loss": val_loss, "test_loss": test_loss,
        "train_acc": train_acc, "val_acc": val_acc, "test_acc": test_acc,
        "train_auc": train_auc, "val_auc": val_metrics["roc_auc"], "test_auc": test_metrics["roc_auc"],
        "val_pr_auc": val_metrics["pr_auc"], "test_pr_auc": test_metrics["pr_auc"], 
        "val_f1": val_metrics["f1"],  "test_f1": test_metrics["f1"], "val_macro_f1": val_metrics["macro_f1"], "test_macro_f1": test_metrics["macro_f1"],
        "val_precision": val_metrics["precision"], "val_recall": val_metrics["recall"], "val_npv": val_metrics["npv"],
        "test_precision": test_metrics["precision"], "test_recall": test_metrics["recall"], "test_npv": test_metrics["npv"],
        "val_specificity": val_metrics["specificity"], "test_specificity": test_metrics["specificity"], 
        "val_sens_at_spec_90": val_metrics['sens_at_spec'], "test_sens_at_spec_90": test_metrics['sens_at_spec'],
        "avg_entropy": test_uncert['avg_entropy'], "entropy_std": test_uncert['entropy_std'],
        "avg_uncertainty": test_uncert['avg_uncertainty'], "uncertainty_std": test_uncert['uncertainty_std'],
        "entropy_class0_avg": test_uncert['entropy_class0_avg'], "entropy_class0_std": test_uncert['entropy_class0_std'], "entropy_class1_avg": test_uncert['entropy_class1_avg'], 
        "entropy_class1_std": test_uncert['entropy_class1_std'], "uncertainty_class0_avg": test_uncert['uncertainty_class0_avg'], "uncertainty_class0_std": test_uncert['uncertainty_class0_std'], 
        "uncertainty_class1_avg": test_uncert['uncertainty_class1_avg'], "uncertainty_class1_std": test_uncert['uncertainty_class1_std'],
        "tn": test_metrics["tn"], "fp": test_metrics["fp"], "fn": test_metrics["fn"], "tp": test_metrics["tp"], "n_samples": test_metrics["n_samples"],
    }

    append_metrics_to_csv(log_file, row)
    
    print_epoch_summary(
        epoch,
        train_loss,
        val_loss,
        test_loss,
        train_auc,
        val_auc,
        test_auc,
        train_acc,
        val_acc,
        test_acc,
        test_metrics,
        test_uncert
    )
    

In [ ]:
models_path = "ResNet34_neh_reduced_GradAug_with_resize_speckle_finetune_ucsd"
# models_path = "ResNet34_neh_reduced_5e4_GradAug_withjust_with_resize_speckle_fintune_octc8_20"

os.makedirs(models_path, exist_ok=True)
log_file = os.path.join(models_path, "training_log.csv")

BEST_MACRO_F1 = -1.0
WARMUP_EPOCHS = 3          # IMPORTANT
LAMBDA_KL = 0.03

checkpoint_path = "grad_aug_resize_speckle_11.pth"
# checkpoint_path = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_resize_Modified/epoch_9.pth"
checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

model.load_state_dict(checkpoint["state_dict"])
model.to(DEVICE)

BEST_MACRO_F1 = -1.0

NUM_EPOCHS = 20

for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(
    model.fc.parameters(),
    lr=1e-3,              # higher LR is safe here
    weight_decay=1e-4
)



for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")

    # -------------------------
    # Train
    # -------------------------
    train_loss, train_auc, train_acc = train_head_only(
        model,
        train_loader,
        optimizer,
        DEVICE,
        criterion,
        train_transform,
    )

    # -------------------------
    # Validate
    # -------------------------

    # -------------------------
    # Validate
    # -------------------------
    val_loss, val_auc, val_acc, val_macro_f1 = validate_head_only(
        model,
        val_loader,
        DEVICE,
        criterion,
        val_transform,
    )

    # test_loss, test_metrics, test_acc, test_auc, test_probs, test_targets, test_uncert = test(
    #     model=model,
    #     loader=test_loader,
    #     device=DEVICE,
    #     criterion=criterion,
    #     val_transform=test_transform
    # )

    scheduler.step()

    # -------------------------
    # Save checkpoint
    # -------------------------
    ckpt = {
        "epoch": epoch,
        "state_dict": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "val_auc": val_auc
    }

    torch.save(ckpt, os.path.join(models_path, f"epoch_{epoch}.pth"))

    if val_macro_f1 > BEST_MACRO_F1:
        BEST_MACRO_F1 = val_macro_f1
        torch.save(ckpt, os.path.join(models_path, "best_model.pth"))

    # -------------------------
    # CSV logging
    # -------------------------
    row = {
        "epoch": epoch,
        "train_loss": train_loss, "val_loss": val_loss,
        "train_acc": train_acc, "val_acc": val_acc,
        "train_auc": train_auc, "val_auc": val_auc, 
        "val_macro_f1": val_macro_f1,  "val_macro_f1": val_macro_f1
    }

    append_metrics_to_csv(log_file, row)
    
    print_epoch_summary(
        epoch,
        train_loss,
        val_loss,
        0.0,  # test_loss is not computed in this version
        train_auc,
        val_auc,
        0.0,  # test_auc is not computed in this version
        train_acc,
        val_acc,
        val_macro_f1
    )
    
    

# Plots

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import pickle
from PIL import Image, ImageFile
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.isotonic import IsotonicRegression
from typing import Tuple

# =====================================================
# 1. VENN-ABERS CLASS DEFINITION
# =====================================================


class VennAbersBinary:
    def __init__(self):
        self.calib_scores = None
        self.calib_labels = None
        self._fitted = False

    def fit(self, scores: np.ndarray, labels: np.ndarray):
        self.calib_scores = np.asarray(scores, dtype=np.float64)
        self.calib_labels = np.asarray(labels, dtype=np.int32)
        self._fitted = True

    def _fit_isotonic(self, scores, labels):
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(scores, labels)
        return ir

    def predict_interval(self, p: float) -> Tuple[float, float]:
        if not self._fitted:
            raise RuntimeError("Call fit() before predict_interval()")
        scores0 = np.append(self.calib_scores, p)
        labels0 = np.append(self.calib_labels, 0)
        ir0 = self._fit_isotonic(scores0, labels0)
        p0 = float(ir0.predict([p])[0])
        scores1 = np.append(self.calib_scores, p)
        labels1 = np.append(self.calib_labels, 1)
        ir1 = self._fit_isotonic(scores1, labels1)
        p1 = float(ir1.predict([p])[0])
        return min(p0, p1), max(p0, p1)

    def predict(self, p: float, method: str = "mean") -> float:
        p0, p1 = self.predict_interval(p)
        if method == "mean": return 0.5 * (p0 + p1)
        return p0 if method == "lower" else p1

# =====================================================
# 2. PATHS & CONFIG
# =====================================================
CHECKPOINT_PATH = "grad_aug_resize_speckle_11.pth"
# CHECKPOINT_PATH = "/media/miglab/DATA_20TB1/OCT/ResNet34_neh_reduced_GradAug_with_resize_speckle_finetune_ucsd/best_model.pth"
SAVE_DIR = "/media/miglab/DATA_20TB1/OCT/ResNet34_neh_reduced_GradAug_with_resize_speckle_finetune_neh"
# CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_with_resize_fintune_octc8_20/best_model.pth"
# SAVE_DIR = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_with_resize_fintune_octc8_20"
SAVE_PATH = os.path.join(SAVE_DIR, "venn_abers_fitted.pkl")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
NUM_CLASSES = 2
ImageFile.LOAD_TRUNCATED_IMAGES = True

# =====================================================
# 3. GRADAUG MODEL DEFINITION
# =====================================================
class ResNet34GradAug(nn.Module):
    def __init__(self, num_classes=2, pretrained=False):
        super().__init__()
        base = models.resnet34(pretrained=pretrained)
        self.conv1, self.bn1, self.relu, self.maxpool = base.conv1, base.bn1, base.relu, base.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = base.layer1, base.layer2, base.layer3, base.layer4
        self.avgpool = base.avgpool
        self.fc = nn.Linear(512, num_classes)

    def forward_full(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)


# =====================================================
# 5. SCORE EXTRACTION
# =====================================================

print(f"--- Loading GradAug Model: {os.path.basename(CHECKPOINT_PATH)} ---")
model = ResNet34GradAug(num_classes=NUM_CLASSES, pretrained=False).to(DEVICE)
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"], strict=True)
model.eval()

val_probs, val_labels = [], []

print("Extracting validation probabilities for calibration...")
with torch.no_grad():
    for imgs_pil, labels in tqdm(val_loader):
        imgs = torch.stack([test_transform(img) for img in imgs_pil]).to(DEVICE)
        logits = model.forward_full(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]
        val_probs.extend(probs.cpu().numpy())
        val_labels.extend(labels.numpy())

# =====================================================
# 6. FIT & SAVE
# =====================================================
va = VennAbersBinary()
va.fit(scores=np.array(val_probs), labels=np.array(val_labels))

os.makedirs(SAVE_DIR, exist_ok=True)
with open(SAVE_PATH, 'wb') as f:
    pickle.dump(va, f)

print(f"\n--- SUCCESS ---")
print(f"Venn-Abers model saved to: {SAVE_PATH}")

In [ ]:
from sklearn.metrics import f1_score

f1_macro = f1_score(val_labels, (np.array(val_probs) >= 0.5).astype(int), average="macro")
f1_macro


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import pickle
from PIL import Image, ImageFile
from tqdm import tqdm

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score, fbeta_score
)
from sklearn.isotonic import IsotonicRegression
from typing import Tuple

# =====================================================
# 1. VENN-ABERS CLASS DEFINITION
# =====================================================
class VennAbersBinary:
    def __init__(self):
        self.calib_scores = None
        self.calib_labels = None
        self._fitted = False

    def fit(self, scores: np.ndarray, labels: np.ndarray):
        self.calib_scores = np.asarray(scores, dtype=np.float64)
        self.calib_labels = np.asarray(labels, dtype=np.int32)
        self._fitted = True

    def _fit_isotonic(self, scores, labels):
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(scores, labels)
        return ir

    def predict_interval(self, p: float) -> Tuple[float, float]:
        if not self._fitted:
            raise RuntimeError("Call fit() before predict_interval()")
        scores0 = np.append(self.calib_scores, p)
        labels0 = np.append(self.calib_labels, 0)
        ir0 = self._fit_isotonic(scores0, labels0)
        p0 = float(ir0.predict([p])[0])
        scores1 = np.append(self.calib_scores, p)
        labels1 = np.append(self.calib_labels, 1)
        ir1 = self._fit_isotonic(scores1, labels1)
        p1 = float(ir1.predict([p])[0])
        return min(p0, p1), max(p0, p1)

    def predict(self, p: float, method: str = "mean") -> float:
        p0, p1 = self.predict_interval(p)
        if method == "mean": return 0.5 * (p0 + p1)
        elif method == "lower": return p0
        elif method == "upper": return p1
        else: raise ValueError("method must be 'mean', 'lower', or 'upper'")

    def predict_batch(self, probs: np.ndarray, method: str = "mean") -> np.ndarray:
        probs = np.asarray(probs, dtype=np.float64)
        return np.array([self.predict(p, method=method) for p in probs])

CHECKPOINT_PATH = "grad_aug_resize_speckle_11.pth"
# CHECKPOINT_PATH = "/media/miglab/DATA_20TB1/OCT/ResNet34_neh_reduced_GradAug_with_resize_speckle_finetune_ucsd/best_model.pth"
VA_PICKLE_PATH = "/media/miglab/DATA_20TB1/OCT/ResNet34_neh_reduced_GradAug_with_resize_speckle_finetune_neh/venn_abers_fitted.pkl"
# CHECKPOINT_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_with_resize_fintune_octc8_20/best_model.pth"
# VA_PICKLE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_GradAug_withjust_with_resize_fintune_octc8_20/venn_abers_fitted.pkl"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE = 512
NUM_CLASSES = 2
ImageFile.LOAD_TRUNCATED_IMAGES = True

# =====================================================
# 3. METRIC HELPERS
# =====================================================

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)
    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)
    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

# def expected_calibration_error(y_true, y_proba, n_bins=10):
#     y_true = np.asarray(y_true, dtype=int)
#     y_proba = np.asarray(y_proba, dtype=np.float64)
#     y_pred = (y_proba >= 0.5).astype(int)
#     confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)
#     bins = np.linspace(0.0, 1.0, n_bins + 1)
#     bin_ids = np.digitize(confidence, bins) - 1
#     ece = 0.0
#     for b in range(n_bins):
#         mask = bin_ids == b
#         if not np.any(mask): continue
#         acc_b = np.mean(y_pred[mask] == y_true[mask])
#         conf_b = np.mean(confidence[mask])
#         ece += np.abs(acc_b - conf_b) * np.mean(mask)
#     return float(ece)



def expected_calibration_error(y_true, y_proba, n_bins=15):
    """
    Standard ECE for binary classification (Guo et al., 2017)
    """
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    # Prediction and confidence
    y_pred = (y_proba >= 0.5).astype(int)
    conf = np.maximum(y_proba, 1.0 - y_proba)

    # Bin edges: confidence ∈ [0.5, 1.0]
    bin_edges = np.linspace(0.5, 1.0, n_bins + 1)

    ece = 0.0
    n = len(y_true)

    for i in range(n_bins):
        bin_lower = bin_edges[i]
        bin_upper = bin_edges[i + 1]

        mask = (conf > bin_lower) & (conf <= bin_upper)
        if not np.any(mask):
            continue

        acc_bin = np.mean(y_pred[mask] == y_true[mask])
        conf_bin = np.mean(conf[mask])
        ece += (np.sum(mask) / n) * abs(acc_bin - conf_bin)

    return float(ece)


# =====================================================
# 5. GRADAUG MODEL DEFINITION
# =====================================================
class ResNet34GradAug(nn.Module):
    def __init__(self, num_classes=2, pretrained=False):
        super().__init__()
        base = models.resnet34(pretrained=pretrained)
        self.conv1, self.bn1, self.relu, self.maxpool = base.conv1, base.bn1, base.relu, base.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = base.layer1, base.layer2, base.layer3, base.layer4
        self.avgpool = base.avgpool
        self.fc = nn.Linear(512, num_classes)

    def forward_full(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

# =====================================================
# 2. INFERENCE (Using GradAug forward_full & existing test_loader)
# =====================================================
print(f"\n--- Loading GradAug Model: {os.path.basename(CHECKPOINT_PATH)} ---")
# Assuming ResNet34GradAug is already defined as in your snippet
model = ResNet34GradAug(num_classes=2, pretrained=False).to(DEVICE)
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"], strict=True)
model.eval()

with open(VA_PICKLE_PATH, 'rb') as f:
    va = pickle.load(f)

probs_all, labels_all = [], []

with torch.no_grad():
    for imgs_pil, labels in tqdm(test_loader, desc="Inference"):
        imgs = torch.stack([test_transform(img) for img in imgs_pil]).to(DEVICE)
        # Using GradAug's specific full forward pass method
        logits = model.forward_full(imgs)
        probs = F.softmax(logits, dim=1)[:, 1]
        probs_all.extend(probs.cpu().numpy())
        labels_all.extend(labels.numpy())

y_true = np.array(labels_all)
y_prob_raw = np.array(probs_all)

print("\nApplying Venn-Abers Calibration...")
y_prob_va = va.predict_batch(y_prob_raw, method="mean")

# =====================================================
# SAVE GRADAUG TEST PREDICTIONS (AFTER VA)
# =====================================================

import os

# Convert calibrated probabilities to predictions
y_pred_va = (y_prob_va >= 0.5).astype(int)

# Dataset order preserved because shuffle=False
test_df_reset = test_loader.dataset.df.reset_index(drop=True)

image_paths = test_df_reset["new_file_path"].values

# Safety check
assert len(image_paths) == len(y_pred_va), "Mismatch between images and predictions!"

df_save = pd.DataFrame({
    "image_path": image_paths,
    "predicted": y_pred_va,
    "ground_truth": y_true,
    "probability_raw": y_prob_raw,
    "probability_va": y_prob_va
})

SAVE_PRED_PATH = os.path.join(
    os.path.dirname(CHECKPOINT_PATH),
    "gradaug_test_predictions_ucsd_with_VA.csv"
)

df_save.to_csv(SAVE_PRED_PATH, index=False)

print(f"\nSaved GradAug VA-calibrated test predictions to:\n{SAVE_PRED_PATH}")
# # =====================================================
# # 3. ORDERED METRICS CALCULATION
# # =====================================================
# def get_ordered_metrics(y_true, y_prob):
#     y_pred = (y_prob >= 0.5).astype(int)
#     conf_stats = set_size_and_singleton(y_prob, tau=0.9)
    
#     # Strictly following the requested order
#     return {
#         "F1 Macro": f1_score(y_true, y_pred, average="macro"),
#         "F2 Macro": fbeta_score(y_true, y_pred, beta=2.0, average="macro", zero_division=0),
#         "F2 Weighted": fbeta_score(y_true, y_pred, beta=2.0, average="weighted", zero_division=0),
#         "Accuracy": accuracy_score(y_true, y_pred),
#         "AUC": roc_auc_score(y_true, y_prob),
#         "Specificity": specificity_score(y_true, y_pred),
#         "Precision Macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
#         "Recall Macro": recall_score(y_true, y_pred, average="macro"),
#         "Precision": precision_score(y_true, y_pred, zero_division=0),
#         "Sensitivity": recall_score(y_true, y_pred),
#         "NPV": npv_score(y_true, y_pred),
#         "Kappa": cohen_kappa_score(y_true, y_pred),
#         "ECE": expected_calibration_error(y_true, y_prob),
#         "Set Size": conf_stats["avg_set_size"],
#         "Singleton": conf_stats["singleton_rate"]
#     }

# metrics_raw = get_ordered_metrics(y_true, y_prob_raw)
# metrics_va = get_ordered_metrics(y_true, y_prob_va)

# # =====================================================
# # 4. FINAL DISPLAY
# # =====================================================
# print("\n" + "=" * 75)
# print(f"GradAug EVALUATION: {os.path.basename(CHECKPOINT_PATH)}")
# print("=" * 75)
# print(f"{'Metric':<30} | {'Before VA':<15} | {'After VA':<15}")
# print("-" * 75)

# for key in metrics_raw.keys():
#     suffix = "%" if key == "Singleton" else ""
#     print(f"{key:<30} | {metrics_raw[key]:<15.4f}{suffix} | {metrics_va[key]:<15.4f}{suffix}")

# print("=" * 75)